# BAMAT207: Probability and Statistics
## Project Activity: Probabilistic SMS Spam Detection
**Topic:** Probability, Random Variables, and Naive Bayes Classification  
**Author / Student ID:** 25BCE2460  
**Deadline:** 27.10.2026  

---
### 📌 Project Overview & Scenario
Develop a Python-based intelligent system that classifies an incoming SMS message as:
- **Spam** ($Y = 1$)
- **Genuine Message (Ham)** ($Y = 0$)

The system rigorously applies fundamental principles of:
1. **Empirical Probability & Prior Probabilities** ($P(\text{Spam})$, $P(\text{Ham})$)
2. **Conditional Probability & Bayes' Theorem** ($P(\text{Spam} \mid \text{Word})$)
3. **Discrete Random Variable Analysis** (Bernoulli Distribution, Expectation $E[Y]$, Variance $\text{Var}(Y)$)
4. **Multinomial Naive Bayes Machine Learning Classification** ($P(\text{Spam} \mid \text{Message})$)
5. **Real-time SMS probability inference**

---
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lucifer-ken07/probabilistic-sms-spam-detection/blob/main/Probabilistic_SMS_Spam_Detection.ipynb)


## 0. Environment Setup & Library Imports
We load the required Python libraries for dataset manipulation (`pandas`), mathematical probability calculations (`numpy`), visualization (`matplotlib`), and machine learning (`scikit-learn`).


In [ ]:
# Install ucimlrepo if not already present
!pip install ucimlrepo -q

import os
import re
import io
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Plot formatting
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("All libraries imported successfully!")


## 1. Activity 1: Dataset Handling
**Tasks:**
- Load SMS messages and labels from the **UCI Machine Learning Repository** (SMS Spam Collection dataset, 5,574 messages).
- Check for missing/null values across columns.
- Remove duplicate messages to prevent data leakage and skewed probabilities.
- Convert textual labels into numerical values:
  $$\text{Spam} = 1, \quad \text{Ham} = 0$$


In [ ]:
# Load SMS Spam Collection from UCI Repository
dataset_url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

try:
    from ucimlrepo import fetch_ucirepo
    print("Attempting to fetch via ucimlrepo...")
    sms_data = fetch_ucirepo(id=228)
    df_raw = pd.DataFrame({
        'label': sms_data.data.targets.iloc[:, 0],
        'message': sms_data.data.features.iloc[:, 0]
    })
    print("Successfully imported via ucimlrepo.")
except Exception as e:
    print(f"ucimlrepo note: {e}. Downloading directly from UCI archive...")
    req = urllib.request.Request(dataset_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response:
        z = zipfile.ZipFile(io.BytesIO(response.read()))
        df_raw = pd.read_csv(
            z.open("SMSSpamCollection"), sep="\t", header=None, names=["label", "message"], encoding="utf-8"
        )
    print("Successfully downloaded and extracted from UCI archive.")

print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()


In [ ]:
# Check missing values
print("--- Missing Values Check ---")
print(df_raw.isnull().sum())

# Check duplicate messages
duplicates = df_raw.duplicated(subset=['message']).sum()
print(f"\nDuplicate messages detected: {duplicates} ({duplicates / len(df_raw) * 100:.2f}%)")

# Remove duplicate messages
df = df_raw.drop_duplicates(subset=['message']).reset_index(drop=True)
print(f"Cleaned dataset shape: {df.shape}")

# Convert labels into numerical values: Spam = 1, Ham = 0
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
df['char_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda msg: len(msg.split()))

print("\nSample cleaned records:")
df[['label', 'label_num', 'char_length', 'word_count', 'message']].head()


## 2. Activity 2: Dataset Analysis & Prior Probabilities
**Tasks:**
- Calculate the total count of genuine (Ham) and Spam messages.
- Calculate prior probabilities:
  $$P(\text{Spam}) = \frac{\text{Number of spam messages}}{\text{Total messages}}$$
  $$P(\text{Ham}) = \frac{\text{Number of ham messages}}{\text{Total messages}}$$
  Note that by Kolmogorov's axioms: $P(\text{Ham}) + P(\text{Spam}) = 1$.
- Display the class distribution using a bar chart and pie chart.


In [ ]:
total_messages = len(df)
class_counts = df['label'].value_counts()
ham_count = class_counts['ham']
spam_count = class_counts['spam']

p_ham = ham_count / total_messages
p_spam = spam_count / total_messages

print("=" * 55)
print("           DATASET CLASS DISTRIBUTION & PRIORS           ")
print("=" * 55)
print(f"Total Messages (N)           : {total_messages}")
print(f"Genuine Messages (Ham)       : {ham_count} ({p_ham * 100:.2f}%)")
print(f"Spam Messages                : {spam_count} ({p_spam * 100:.2f}%)")
print("-" * 55)
print(f"Prior Probability P(Ham)     : {p_ham:.4f}")
print(f"Prior Probability P(Spam)    : {p_spam:.4f}")
print(f"Sum of Priors                : {p_ham + p_spam:.4f}")
print("=" * 55)


In [ ]:
# Visualization: Class Distribution Bar & Pie Charts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), dpi=120)

colors = ['#2b5c8f', '#d9534f']
bars = ax1.bar(['Ham (0)', 'Spam (1)'], [ham_count, spam_count], color=colors, width=0.5, edgecolor='black', linewidth=1.2)
ax1.set_title("Class Frequency Distribution", fontsize=13, fontweight='bold', pad=12)
ax1.set_ylabel("Number of Messages", fontsize=11)
ax1.set_ylim(0, max(ham_count, spam_count) * 1.15)
ax1.grid(axis='y', linestyle='--', alpha=0.6)

for bar, count, p in zip(bars, [ham_count, spam_count], [p_ham, p_spam]):
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 50, f"{count:,}\n({p*100:.1f}%)",
             ha='center', va='bottom', fontsize=11, fontweight='bold')

ax2.pie([ham_count, spam_count], labels=['Ham', 'Spam'], autopct='%1.2f%%',
        startangle=140, colors=colors, explode=(0.04, 0.08),
        wedgeprops=dict(width=0.45, edgecolor='black'))
ax2.set_title("Prior Probabilities P(Class)", fontsize=13, fontweight='bold', pad=12)

plt.tight_layout()
plt.show()


## 3. Activity 3: Word-Based Probability Analysis & Bayes' Theorem Verification
**Tasks:**
- Select 5 suspicious words commonly observed in fraudulent SMS: `free`, `prize`, `win`, `urgent`, `offer`.
- Calculate the empirical conditional probability:
  $$P(\text{Spam} \mid \text{Word}) = \frac{\text{Number of spam messages containing Word}}{\text{Total messages containing Word}} = \frac{N(\text{Spam} \cap \text{Word})}{N(\text{Word})}$$
- Verify the result using **Bayes' Theorem**:
  $$P(\text{Spam} \mid \text{Word}) = \frac{P(\text{Word} \mid \text{Spam}) \times P(\text{Spam})}{P(\text{Word})}$$
  where:
  - $P(\text{Word} \mid \text{Spam}) = \frac{N(\text{Spam} \cap \text{Word})}{N(\text{Spam})}$ (Likelihood)
  - $P(\text{Spam}) = \frac{N(\text{Spam})}{N_{\text{total}}}$ (Prior)
  - $P(\text{Word}) = \frac{N(\text{Word})}{N_{\text{total}}}$ (Marginal Evidence)


In [ ]:
suspicious_words = ['free', 'prize', 'win', 'urgent', 'offer']

word_analysis = []
n_total = len(df)
n_spam_total = (df['label_num'] == 1).sum()
n_ham_total = (df['label_num'] == 0).sum()

for word in suspicious_words:
    pattern = rf"\b{re.escape(word)}\b"
    has_word = df['message'].str.contains(pattern, case=False, regex=True)
    
    count_w = has_word.sum()
    count_w_spam = (has_word & (df['label_num'] == 1)).sum()
    count_w_ham = (has_word & (df['label_num'] == 0)).sum()
    
    # Direct empirical conditional probability
    p_spam_given_w = count_w_spam / count_w if count_w > 0 else 0.0
    p_ham_given_w = count_w_ham / count_w if count_w > 0 else 0.0
    
    # Likelihoods and marginal probability
    p_w_given_spam = count_w_spam / n_spam_total if n_spam_total > 0 else 0.0
    p_w_given_ham = count_w_ham / n_ham_total if n_ham_total > 0 else 0.0
    p_w = count_w / n_total
    
    # Bayes' Theorem formulation
    p_bayes_spam = (p_w_given_spam * p_spam) / p_w if p_w > 0 else 0.0
    diff = abs(p_spam_given_w - p_bayes_spam)
    
    word_analysis.append({
        'Word': word,
        'Count(W)': count_w,
        'Spam ∩ W': count_w_spam,
        'Ham ∩ W': count_w_ham,
        'P(W|Spam)': round(p_w_given_spam, 4),
        'P(W)': round(p_w, 4),
        'P(Spam|W) [Direct]': round(p_spam_given_w, 4),
        'P(Spam|W) [Bayes]': round(p_bayes_spam, 4),
        'Abs Difference': f"{diff:.1e}"
    })

df_words = pd.DataFrame(word_analysis)
display(df_words)


In [ ]:
# Visualization of Word Conditional Probabilities
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
words = df_words['Word']
p_spam_cond = df_words['P(Spam|W) [Direct]']
p_ham_cond = 1.0 - p_spam_cond

x = np.arange(len(words))
width = 0.35

rects1 = ax.bar(x - width/2, p_spam_cond, width, label='P(Spam | Word)', color='#d9534f', edgecolor='black')
rects2 = ax.bar(x + width/2, p_ham_cond, width, label='P(Ham | Word)', color='#2b5c8f', edgecolor='black')

ax.set_title("Conditional Probabilities of Suspicious Words", fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel("Suspicious Keywords", fontsize=11)
ax.set_ylabel("Conditional Probability", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(words, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.18)
ax.legend(loc='upper right', frameon=True)
ax.grid(axis='y', linestyle='--', alpha=0.5)

for rect in rects1:
    h = rect.get_height()
    ax.annotate(f"{h:.2f}", xy=(rect.get_x() + rect.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight='bold')

for rect in rects2:
    h = rect.get_height()
    ax.annotate(f"{h:.2f}", xy=(rect.get_x() + rect.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


## 4. Activity 4: Random Variable Analysis
**Definition:**  
Let $Y$ be a discrete indicator random variable modeling the spam status of an incoming SMS:
$$Y = \begin{cases} 1, & \text{if message is Spam} \\ 0, & \text{if message is Genuine (Ham)} \end{cases}$$

Since $Y$ takes on two values $\{0, 1\}$, it follows a **Bernoulli Distribution**:
$$Y \sim \text{Bernoulli}(p), \quad \text{where } p = P(Y = 1) = P(\text{Spam})$$

### Theoretical Derivations:
1. **Probability Mass Function (PMF):**
   $$P(Y = y) = p^y (1 - p)^{1 - y}, \quad y \in \{0, 1\}$$

2. **Mathematical Expectation $E[Y]$:**
   $$E[Y] = \sum_{y \in \{0, 1\}} y \cdot P(Y = y) = 0 \cdot (1 - p) + 1 \cdot p = p = P(\text{Spam})$$

3. **Variance $\text{Var}(Y)$:**
   $$\text{Var}(Y) = E[Y^2] - (E[Y])^2 = (0^2 \cdot (1 - p) + 1^2 \cdot p) - p^2 = p - p^2 = p(1 - p)$$

4. **Standard Deviation $\sigma(Y)$:**
   $$\sigma(Y) = \sqrt{\text{Var}(Y)} = \sqrt{p(1 - p)}$$


In [ ]:
y_values = df['label_num'].values
n = len(y_values)
p = p_spam

# Theoretical values
e_y_theoretical = p
var_y_theoretical = p * (1 - p)
std_y_theoretical = np.sqrt(var_y_theoretical)

# Empirical sample values
sample_mean = np.mean(y_values)
sample_var = np.var(y_values, ddof=1)
sample_std = np.std(y_values, ddof=1)

rv_table = pd.DataFrame({
    'Metric': ['Parameter p = P(Spam)', 'Expectation E(Y)', 'Variance Var(Y)', 'Standard Deviation σ(Y)'],
    'Theoretical Formula': ['p = N_spam / N', 'p', 'p * (1 - p)', 'sqrt(p * (1 - p))'],
    'Theoretical Value': [p, e_y_theoretical, var_y_theoretical, std_y_theoretical],
    'Empirical Sample Value': [sample_mean, sample_mean, sample_var, sample_std]
})
display(rv_table)


In [ ]:
# Plot PMF of Random Variable Y
fig, ax = plt.subplots(figsize=(7, 4.5), dpi=120)
bars = ax.bar([0, 1], [1 - p, p], width=0.4, color=['#2b5c8f', '#d9534f'], edgecolor='black', linewidth=1.2)

ax.set_title(f"PMF of Bernoulli Random Variable Y ~ Bern(p = {p:.4f})", fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel("Random Variable Outcome y (0 = Ham, 1 = Spam)", fontsize=11)
ax.set_ylabel("P(Y = y)", fontsize=11)
ax.set_xticks([0, 1])
ax.set_xticklabels(['0 (Ham)', '1 (Spam)'], fontsize=11)
ax.set_ylim(0, 1.1)
ax.grid(axis='y', linestyle='--', alpha=0.5)

for bar, prob in zip(bars, [1 - p, p]):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, h + 0.03, f"{prob:.4f}",
            ha='center', va='bottom', fontsize=11, fontweight='bold')

summary_box = f"E(Y) = {e_y_theoretical:.4f}\nVar(Y) = {var_y_theoretical:.4f}\nσ(Y) = {std_y_theoretical:.4f}"
ax.text(0.7, 0.75, summary_box, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()


## 5. Activity 5: Data Preparation
**Tasks:**
- Split data into **80% Training** and **20% Testing** sets.
- Apply **Stratified Sampling** (`stratify=y`) so both subsets reflect the exact $12.63\%$ spam prevalence.
- Extract word features using `CountVectorizer` (Bag-of-Words representation).


In [ ]:
X = df['message']
y = df['label_num']

# 80/20 Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Total dataset size : {len(df)}")
print(f"Training set size  : {len(X_train)} (80%)")
print(f"Test set size      : {len(X_test)} (20%)")
print(f"Train spam proportion: {(y_train == 1).mean() * 100:.2f}%")
print(f"Test spam proportion : {(y_test == 1).mean() * 100:.2f}%")

# Feature extraction: CountVectorizer
vectorizer = CountVectorizer(stop_words='english', lowercase=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"\nVocabulary Size: {len(vectorizer.vocabulary_)} distinct word tokens")
print(f"X_train feature matrix shape: {X_train_vec.shape}")
print(f"X_test feature matrix shape : {X_test_vec.shape}")


## 6. Activity 6: Model Building (Multinomial Naive Bayes)
### Mathematical Foundation:
The **Multinomial Naive Bayes (MNB)** classifier is ideally suited for discrete word count features $x = (x_1, x_2, \dots, x_d)$.

Under the Naive Bayes assumption, word counts are conditionally independent given the class label $c \in \{0, 1\}$:
$$P(x_1, x_2, \dots, x_d \mid Y = c) = \prod_{j=1}^d P(w_j \mid Y = c)^{x_j}$$

With **Laplace Smoothing** ($\alpha = 1$):
$$\hat{\theta}_{cj} = \frac{N_{cj} + \alpha}{N_c + \alpha \cdot d}$$
where $N_{cj}$ is the count of word $j$ in class $c$, and $d$ is the vocabulary size.

The classification decision rule maximizes the posterior:
$$\hat{y} = \arg\max_{c \in \{0, 1\}} \left[ \log P(Y = c) + \sum_{j=1}^d x_j \log \hat{\theta}_{cj} \right]$$


In [ ]:
# Train Multinomial Naive Bayes classifier
model = MultinomialNB(alpha=1.0)
model.fit(X_train_vec, y_train)

print("Model successfully trained!")
print(f"Log Prior Probabilities [Ham, Spam]: {model.class_log_prior_}")
print(f"Estimated Prior P(Ham) : {np.exp(model.class_log_prior_[0]):.4f}")
print(f"Estimated Prior P(Spam): {np.exp(model.class_log_prior_[1]):.4f}")


## 7. Activity 7: Model Evaluation
**Tasks:**
- Evaluate model using:
  - **Accuracy**
  - **Precision**
  - **Recall**
  - **F1-score**
  - **Confusion Matrix**
- Specifically identify and analyze:
  - **False Negatives**: Spam messages incorrectly classified as genuine
  - **False Positives**: Genuine messages incorrectly classified as spam


In [ ]:
y_pred = model.predict(X_test_vec)
y_proba = model.predict_proba(X_test_vec)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("=" * 50)
print("             MODEL EVALUATION METRICS             ")
print("=" * 50)
print(f"Accuracy  : {acc * 100:.2f}%")
print(f"Precision : {prec * 100:.2f}%")
print(f"Recall    : {rec * 100:.2f}%")
print(f"F1-Score  : {f1 * 100:.2f}%")
print("=" * 50)
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham (0)', 'Spam (1)'], digits=4))


In [ ]:
# Confusion Matrix Heatmap
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5), dpi=120)
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im)

classes = ['Ham (0)', 'Spam (1)']
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(classes, fontsize=11, fontweight='bold')
ax.set_yticklabels(classes, fontsize=11, fontweight='bold')

labels = [
    [f"TN\n{tn}\n({tn/(tn+fp)*100:.1f}%)", f"FP\n{fp}\n({fp/(tn+fp)*100:.1f}%)"],
    [f"FN\n{fn}\n({fn/(fn+tp)*100:.1f}%)", f"TP\n{tp}\n({tp/(fn+tp)*100:.1f}%)"]
]
thresh = cm.max() / 2.0
for i in range(2):
    for j in range(2):
        ax.text(j, i, labels[i][j], ha='center', va='center',
                color="white" if cm[i, j] > thresh else "black", fontsize=12, fontweight='bold')

ax.set_title(f"Confusion Matrix (Accuracy: {acc*100:.2f}%)", fontsize=12, fontweight='bold', pad=12)
ax.set_ylabel("True Ground Truth Label", fontsize=11)
ax.set_xlabel("Model Predicted Label", fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Qualitative Misclassification Analysis
test_df = pd.DataFrame({
    'Message': X_test,
    'True_Label': y_test,
    'Pred_Label': y_pred,
    'P(Spam|Msg)': y_proba[:, 1],
    'P(Ham|Msg)': y_proba[:, 0]
})

fps = test_df[(test_df['True_Label'] == 0) & (test_df['Pred_Label'] == 1)]
fns = test_df[(test_df['True_Label'] == 1) & (test_df['Pred_Label'] == 0)]

print("=" * 70)
print(f"1. FALSE POSITIVES (Genuine Ham misclassified as Spam): {len(fps)}")
print("=" * 70)
for idx, row in fps.head(4).iterrows():
    print(f"• Message: "{row['Message']}"")
    print(f"  P(Spam|Msg) = {row['P(Spam|Msg)']:.4f} | P(Ham|Msg) = {row['P(Ham|Msg)']:.4f}\n")

print("=" * 70)
print(f"2. FALSE NEGATIVES (Spam misclassified as Genuine Ham): {len(fns)}")
print("=" * 70)
for idx, row in fns.head(4).iterrows():
    print(f"• Message: "{row['Message']}"")
    print(f"  P(Spam|Msg) = {row['P(Spam|Msg)']:.4f} | P(Ham|Msg) = {row['P(Ham|Msg)']:.4f}\n")


## 8. Activity 8: Real-Time Testing
**Task:**
Allow entering a new SMS message and compute:
- $\text{Prediction}$: Spam or Genuine (Ham)
- $P(\text{Spam} \mid \text{Message})$
- $P(\text{Ham} \mid \text{Message})$

We test the exact example provided in the syllabus:
> `"Congratulations! You have won a free prize. Claim now."`


In [ ]:
def predict_sms(message, model=model, vectorizer=vectorizer):
    vec = vectorizer.transform([message])
    pred = model.predict(vec)[0]
    proba = model.predict_proba(vec)[0]
    
    p_ham, p_spam = proba[0], proba[1]
    label = "Spam" if pred == 1 else "Ham (Genuine message)"
    
    print("-" * 65)
    print(f"Input Message : "{message}"")
    print(f"Prediction    : {label}")
    print(f"P(Spam | Msg) : {p_spam:.4f} ({p_spam * 100:.2f}%)")
    print(f"P(Ham | Msg)  : {p_ham:.4f} ({p_ham * 100:.2f}%)")
    print("-" * 65)
    return label, p_spam, p_ham

# Test required assignment benchmark
sample_1 = "Congratulations! You have won a free prize. Claim now."
predict_sms(sample_1)

# Additional test cases
predict_sms("Hey, are we still meeting for lunch at 12:30 today?")
predict_sms("URGENT! Your bank account has been compromised. Verify your details immediately to claim your refund.")
predict_sms("Can you send me the lecture notes for BAMAT207 Probability and Statistics?")


In [ ]:
# Interactive Live Test
custom_input = "Claim your exclusive voucher worth $500 today! Text YES to claim."
predict_sms(custom_input)


## 9. Conclusion & Learning Outcomes
Through this project, we demonstrated:
1. **Prior Probability & Class Imbalance:** Quantifying that only $12.63\%$ of SMS messages are spam, directly influencing Bayesian posterior estimation.
2. **Bayes' Theorem in NLP:** Verified that empirical conditional probabilities $P(\text{Spam} \mid \text{Word})$ match Bayes' theorem $\frac{P(\text{Word} \mid \text{Spam}) \times P(\text{Spam})}{P(\text{Word})}$ with zero mathematical divergence.
3. **Random Variables:** Modeled spam classification as a Bernoulli random variable $Y$, verifying that sample statistics match theoretical expectation $E[Y] = p$ and variance $\text{Var}(Y) = p(1 - p)$.
4. **Multinomial Naive Bayes:** Built an end-to-end classifier achieving **$98.36\%$ accuracy** and **$93.33\%$ F1-score**, effectively guarding against spam while maintaining extremely low false positive rates ($0.55\%$).
